# Bhu-Aadhaar 3D: GPU training v2, human-labelled data first
Team Dimension Zero · SIH26011. Push with `bash scripts/kaggle_train.sh push-full`.

v1 trained the Indian part only on Google/Microsoft *model* footprints, which are misaligned, merge row houses and
mark trees and railway tracks as buildings. v2 adds **ramp** (DevGlobal, CC BY-NC 4.0): 51k Indian (Karnataka) and
South Asian buildings drawn by one person and reviewed by another. Models are selected on human-labelled Indian
validation tiles, and the headline test score is on **held-out human-labelled Karnataka tiles** (whole ~1 km cells held out).

Reuses the drone chips made by the `ulpin3d-train` notebook (attached as input). Only `/kaggle/working/models` is saved.

In [ ]:
S1_EPOCHS, S2_EPOCHS = 25, 16
import os, glob, subprocess, shutil
def find(name, depth=6):
    # Kaggle mounts inputs at /kaggle/input/<slug>/ or /kaggle/input/<kind>/<owner>/<slug>/; search a few levels, not the whole chip tree
    return [p for d in range(1, depth + 1) for p in glob.glob('/kaggle/input/' + '*/' * d + name)]
cands = [os.path.dirname(p) for p in find('pyproject.toml')]
src = [c for c in cands if not os.path.exists(c + '/data/footprint/chips/index.csv')]   # the code dataset, not the old run's copy
prep = [os.path.dirname(os.path.dirname(p)) for p in find('data/footprint/chips_reg/index.csv', 5)]
print('code:', src, '\nchips:', prep)
assert src, 'attach the ulpin3d-engine dataset'
assert prep and os.path.exists(prep[0] + '/chips/index.csv'), 'attach the output of the ulpin3d-train notebook (it holds the chips)'
CHIPS, CHIPS_REG = prep[0] + '/chips', prep[0] + '/chips_reg'
WORK = '/kaggle/temp/ulpin3d-engine'
os.makedirs('/kaggle/temp', exist_ok=True)
if not os.path.exists(WORK):
    subprocess.run(['cp', '-r', src[0], WORK], check=True)
os.chdir(WORK)
OUT = '/kaggle/working/models'
os.makedirs(OUT, exist_ok=True)
!pip install -q -e . 2>&1 | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
import pandas as pd
for n, p in (('chips', CHIPS), ('chips_reg', CHIPS_REG)):
    print(n, pd.read_csv(p + '/index.csv').groupby(['split', 'region']).size().to_dict())

## Human-labelled ramp tiles -> chips (~30 min, needs Internet)

In [ ]:
RAMP = '/kaggle/temp/chips_ramp'
!python -W ignore -m ulpin3d.cli ramp-chips --out {RAMP} --workers 32 | tail -25

## Stage 1: pretrain (human labels weighted up, noisy labels down)

In [ ]:
!python -m ulpin3d.cli train-footprint --config configs/footprint_stage1_ramp.yaml --set "data.chips_root=[{CHIPS}, {RAMP}]" train.epochs={S1_EPOCHS}
shutil.copy('runs/footprint_stage1/best.pt', OUT + '/footprint_stage1.pt')
shutil.copy('runs/footprint_stage1/history.csv', OUT + '/history_stage1.csv')

## Stage 2: adapt to Indian drone imagery

In [ ]:
!python -m ulpin3d.cli train-footprint --config configs/footprint_stage2_india_ramp.yaml --set "data.chips_root=[{CHIPS_REG}, {RAMP}]" train.epochs={S2_EPOCHS}
shutil.copy('runs/footprint_stage2_india/best.pt', OUT + '/footprint_india.pt')
shutil.copy('runs/footprint_stage2_india/history.csv', OUT + '/history_stage2.csv')

## Test
* `*_ramp_india_test`: held-out human-labelled Karnataka tiles (the honest number)
* `*_drone_india_test`: held-out Indian drone scenes, scored against Google/Microsoft labels (noisy reference)

In [ ]:
for name, ck in (('stage1', 'runs/footprint_stage1/best.pt'), ('stage2', 'runs/footprint_stage2_india/best.pt')):
    !python -m ulpin3d.cli eval-footprint --checkpoint {ck} --chips {RAMP}      --split test --out reports/{name}_ramp_india_test  | tail -4
    !python -m ulpin3d.cli eval-footprint --checkpoint {ck} --chips {CHIPS_REG} --split test --out reports/{name}_drone_india_test | tail -8
!python -m ulpin3d.cli export-onnx --checkpoint runs/footprint_stage2_india/best.pt --out models/footprint_india.onnx
!cp models/footprint_india.onnx* {OUT}/
!cp -r reports {OUT}/
!cp {RAMP}/index.csv {OUT}/ramp_index.csv
!ls -la {OUT}